
# Measure image quality

Compute error and similarity measures against a reference image, also during the iterations of an algorithm.

Source: [Python/demos/d10_MeasureQuality.py](https://github.com/CERN/TIGRE/blob/master/Python/demos/d10_MeasureQuality.py)_ · [MATLAB/Demos/d10_QualityMeasurements.m](https://github.com/CERN/TIGRE/blob/master/MATLAB/Demos/d10_QualityMeasurements.m)_

.. dropdown:: MATLAB version of this demo
   :icon: code

   .. literalinclude:: /../MATLAB/Demos/d10_QualityMeasurements.m
      :language: matlab


## Initialize



In [ ]:
import tigre
import numpy as np
from tigre.utilities import sample_loader
from tigre.utilities import CTnoise
import tigre.algorithms as algs
from matplotlib import pyplot as plt
from tigre.utilities.Measure_Quality import Measure_Quality as MQ

## Geometry



In [ ]:
geo = tigre.geometry_default(high_resolution=False)

## Load data and generate projections



In [ ]:
# define angles
angles = np.linspace(0, 2 * np.pi, 100, endpoint=False)
# Load thorax phantom data
head = sample_loader.load_head_phantom(geo.nVoxel)
# generate projections
projections = tigre.Ax(head, geo, angles)
# add noise
noise_projections = CTnoise.add(projections, Poisson=1e5, Gaussian=np.array([0, 10]))

## Image quality measurement



In [ ]:
# There are three inputs to the function

#
#           'RMSE' : The square root of the mean of the squared differences
#                    of pixel intensities of two images.
#
#                    Taken from "A rigid motion correction method for helical
#                    computed tomography (CT)"
#                    (doi:10.1088/0031-9155/60/5/2047)
#           'nRMSE': The square root of the mean of the squared differences
#                    of pixel intensities of two images, normalized
#
#           'CC'   : The Pearson correlation coefficient which measures the
#                    linear dependence between two images.
#
#                    Taken from "A rigid motion correction method for helical
#                    computed tomography (CT)"
#                    (doi:10.1088/0031-9155/60/5/2047)
#
#           'MSSIM': The mean structural similarity index, a measure of the
#                    similarity of two images in terms of luminance,
#                    contrast and structure that is designed to provide a
#                    good approximation of perceptual image quality.
#
#                    Taken from "A rigid motion correction method for helical
#                    computed tomography (CT)"
#                    (doi:10.1088/0031-9155/60/5/2047)
#
#           'UQI' : The universal quality index to evaluate the degree of
#                   similarity between the reconstructed and phantom images
#                   for chosen ROIs. Its value ranges from zero to one.
#                   A UQI value closer to one suggests better similarity to true image.
#
#                   Taken from "Few-view cone-beam CT reconstruction with
#                   deformed prior image" (doi: 10.1118/1.4901265)
#
#
#
#

## Define a cell array of image quality measures
qualmeas = ["RMSE", "CC", "MSSIM", "UQI"]

## Reconstruct image using SIRT, SART and OS-SART



In [ ]:
# =========================================================================
imgSIRT, qualitySIRT = algs.sirt(noise_projections, geo, angles, 20, Quameasopts=qualmeas)
imgSART, qualitySART = algs.sart(noise_projections, geo, angles, 20, Quameasopts=qualmeas)
imgOSSART, qualityOSSART = algs.ossart(noise_projections, geo, angles, 20, Quameasopts=qualmeas)

## Plot the changes of image quality measures over iterations, i.e. how much each iteration looks like the previous iteration.



In [ ]:
# (without the original image one can not compute the absolute image quality, of course)

##RMSE plot
plt.figure()
plt.plot(np.vstack((qualitySIRT[0, :], qualityOSSART[0, :], qualitySART[0, :])).T)
plt.title("Evolution of RMSE per iteration")
plt.xlabel("Iteration")
plt.ylabel("RMSE")
plt.gca().legend(("SIRT", "OS-SART", "SART"))

##CC plot
plt.figure()
plt.plot(np.vstack((qualitySIRT[1, :], qualityOSSART[1, :], qualitySART[1, :])).T)
plt.title("Evolution of CC per iteration")
plt.xlabel("Iteration")
plt.ylabel("CC")
plt.gca().legend(("SIRT", "OS-SART", "SART"))

##MSSIM plot
plt.figure()
plt.plot(np.vstack((qualitySIRT[2, :], qualityOSSART[2, :], qualitySART[2, :])).T)
plt.title("Evolution of MSSIM per iteration")
plt.xlabel("Iteration")
plt.ylabel("MSSIM")
plt.gca().legend(("SIRT", "OS-SART", "SART"))

##UQI plot
plt.figure()
plt.plot(np.vstack((qualitySIRT[3, :], qualityOSSART[3, :], qualitySART[3, :])).T)
plt.title("Evolution of UQI per iteration")
plt.xlabel("Iteration")
plt.ylabel("UQI")
plt.gca().legend(("SIRT", "OS-SART", "SART"))

plt.show()

## Compute the parameters for the target image



In [ ]:
# Knowing change of the parameters per iteration can be interesting, but
# The main use of them would be to compare a reconstructed image to a given
# known target image.

sart_uqi = MQ(head, imgSART, ["UQI"])
sirt_uqi = MQ(head, imgSIRT, ["UQI"])
ossart_uqi = MQ(head, imgOSSART, ["UQI"])

print("UQI for SART,   SIRT,   OS-SART")
print("       " + str(sart_uqi) + " " + str(sirt_uqi) + " " + str(ossart_uqi))

sart_rmse = MQ(head, imgSART, ["RMSE"])
sirt_rmse = MQ(head, imgSIRT, ["RMSE"])
ossart_rmse = MQ(head, imgOSSART, ["RMSE"])

print("RMSE for SART,   SIRT,   OS-SART")
print("       " + str(sart_rmse) + " " + str(sirt_rmse) + " " + str(ossart_rmse))